# Text Preprocessing for 10-K Sections

Produces **two text variants** from the raw extracted sections (item\_1, item\_1A, item\_7):

| Variant | Cleaning | Stemmed | Use in |
|---|---|---|---|
| `*_clean` | HTML strip, whitespace norm, ASCII | No | FinBERT, BERTopic |
| `*_stemmed` | Clean + stopword removal + Snowball | Yes | LM dictionary, TF-IDF cosine sim |

**Also produces** `text_chunks.parquet`: clean text split into 400-token windows with 50-token overlap,
ready for batch FinBERT inference.

**Output files** (written to Drive):
- `text_preprocessed.parquet` — one row per (cik, year), both variants per section
- `text_chunks.parquet` — one row per (cik, year, section, chunk_idx)

## 0. Mount Drive

In [8]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Configuration

In [9]:
import os

CONFIG = {
    # Folder containing {cik}_filings.parquet files (output of SEC_10K_Extractor_Drive)
    'filings_folder': '/content/drive/MyDrive/FML_project_4',

    # Where to write preprocessed outputs
    'output_folder': '/content/drive/MyDrive/FML_project_4',

    # Sections to process
    'sections': ['item_1', 'item_1A', 'item_7'],

    # FinBERT tokenizer used for accurate chunk token counting
    'tokenizer_name': 'ProsusAI/finbert',

    # BERT chunk size (tokens) and overlap
    'chunk_size': 400,
    'chunk_overlap': 50,

    # Hard cap on raw section length to avoid memory issues (chars)
    'max_section_chars': 150_000,

    'start_year': 2004,
    'end_year':   2025,
}
print('Config OK')

Config OK


## 2. Install dependencies

In [ ]:
import subprocess
subprocess.run(
    ['pip', 'install', '-q', 'nltk', 'PyStemmer', 'transformers', 'pyarrow', 'tqdm'],
    check=True,
)

import nltk
nltk.download('stopwords', quiet=True)
print('Dependencies ready')

## 3. Load parquet files

In [11]:
import glob
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
import gc

SECTIONS: list[str] = CONFIG['sections']

pq_files = sorted(
    glob.glob(os.path.join(CONFIG['filings_folder'], '**', '*_filings.parquet'), recursive=True)
)
print(f'Parquet files found: {len(pq_files)}')

Parquet files found: 977


## 4. Cleaning pipeline

Strips HTML tags, normalises whitespace, removes non-ASCII characters, and
collapses repeated punctuation. **Does NOT stem** — the clean text is used
as-is by FinBERT and BERTopic.

In [12]:
import re

_HTML_TAG    = re.compile(r'<[^>]+>')
_HTML_ENTITY = re.compile(r'&[a-zA-Z]{2,6};|&#\d{1,5};')
_NON_ASCII   = re.compile(r'[^\x00-\x7F]+')
_WHITESPACE  = re.compile(r'[ \t]+')
_MULTILINE   = re.compile(r'\n{3,}')
_PAGE_NUM    = re.compile(r'\n\s*\d{1,3}\s*\n')


def clean_series(s: pd.Series) -> pd.Series:
    """Vectorised cleaning using pandas C-level str ops (~10x faster than a Python loop)."""
    return (
        s.str[:CONFIG['max_section_chars']]
         .str.replace(_HTML_TAG,    ' ', regex=True)
         .str.replace(_HTML_ENTITY, ' ', regex=True)
         .str.replace(_NON_ASCII,   ' ', regex=True)
         .str.replace(_PAGE_NUM,    '\n', regex=True)
         .str.replace(_WHITESPACE,  ' ', regex=True)
         .str.replace(_MULTILINE,   '\n\n', regex=True)
         .str.strip()
    )


# Smoke test
_t = pd.Series(['<p>This &amp; that\u2019s fine.\n\n\n\n   </p>'])
assert 'This' in clean_series(_t).iloc[0]
print('clean_series OK')

clean_series OK


## 5. Stemming pipeline

Uses **PyStemmer** (C bindings to Snowball) via `stemmer.stemWords(word_list)` — a single
C call per document instead of one Python call per word. ~10–20x faster than NLTK's
`SnowballStemmer` and requires no worker processes, so no fork-based memory explosion.

In [ ]:
import Stemmer as _PyStemmer
from nltk.corpus import stopwords

_STOPWORDS: frozenset[str] = frozenset(stopwords.words('english'))
# PyStemmer wraps the C Snowball library — stems a whole word list in one call,
# ~10–20x faster than iterating NLTK's SnowballStemmer in Python.
_STEMMER = _PyStemmer.Stemmer('english')
_WORD_RE = re.compile(r'[a-zA-Z]+')


def stem_series(s: pd.Series) -> pd.Series:
    """Stem an entire Series using PyStemmer (C-level batch call, no forking)."""
    out: list[str] = []
    for text in s:
        tokens = [t for t in _WORD_RE.findall(text.lower()) if t not in _STOPWORDS and len(t) > 2]
        out.append(' '.join(_STEMMER.stemWords(tokens)))
    return pd.Series(out, index=s.index)


# Smoke test
_res = stem_series(pd.Series(['The companies are managing their risk factors.']))
assert 'compani' in _res.iloc[0]
print('stem_series OK')

## 6. Streaming pipeline: clean → stem → chunk → save

Processes **50 parquet files per batch**. Each batch is written to a local temp file
(`/content/temp_batches/`) before the next batch starts.

**Resume-safe**: if the kernel crashes or you stop it, re-running this cell skips any
batch whose temp file already exists. A final merge cell copies everything to Drive.

In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq
from collections import defaultdict
from transformers import AutoTokenizer
import glob as _glob

# ── Tokenizer ──
tokenizer = AutoTokenizer.from_pretrained(CONFIG['tokenizer_name'])
CHUNK_SIZE: int  = CONFIG['chunk_size']
STRIDE: int      = CONFIG['chunk_overlap']
DOC_BATCH: int   = 128   # docs per tokenizer call
BATCH_FILES: int = 50    # parquet files per outer iteration

# Temp dir is local SSD — fast writes, survives notebook restarts within the same session
TEMP_DIR = '/content/temp_batches'
os.makedirs(TEMP_DIR, exist_ok=True)

PRE_PATH    = os.path.join(CONFIG['output_folder'], 'text_preprocessed.parquet')
CHUNKS_PATH = os.path.join(CONFIG['output_folder'], 'text_chunks.parquet')
load_cols   = ['cik', 'year', 'filing_date'] + SECTIONS

file_batches = [pq_files[i : i + BATCH_FILES] for i in range(0, len(pq_files), BATCH_FILES)]

# ── Resume: find already-completed batch temp files ──
done_batches: set[int] = {
    int(os.path.basename(f)[4:8])          # pre_0003.parquet → 3
    for f in _glob.glob(f'{TEMP_DIR}/pre_*.parquet')
}
print(f'{len(file_batches)} batches total — {len(done_batches)} already done — {len(file_batches) - len(done_batches)} remaining')

for batch_num, batch_paths in enumerate(tqdm(file_batches, desc='Batches')):
    pre_tmp = f'{TEMP_DIR}/pre_{batch_num:04d}.parquet'
    chk_tmp = f'{TEMP_DIR}/chk_{batch_num:04d}.parquet'

    if os.path.exists(pre_tmp):
        continue   # already processed — skip

    # ── 1. Load ─────────────────────────────────────────────────────────────
    dfs: list[pd.DataFrame] = []
    for p in batch_paths:
        try:
            dfs.append(pd.read_parquet(p, columns=load_cols))
        except Exception as exc:
            print(f'  Skip {p}: {exc}')
    if not dfs:
        open(pre_tmp, 'w').close()   # empty sentinel so we don't retry
        continue

    batch = pd.concat(dfs, ignore_index=True)
    del dfs; gc.collect()

    batch['cik']  = batch['cik'].astype(str).str.strip().str.lstrip('0')
    batch['year'] = pd.to_numeric(batch['year'], errors='coerce').astype('Int64')
    batch = batch[batch['year'].between(CONFIG['start_year'], CONFIG['end_year'])]
    for sec in SECTIONS:
        if sec not in batch.columns:
            batch[sec] = ''
        batch[sec] = batch[sec].fillna('').astype(str)
    batch = (
        batch.sort_values('filing_date', na_position='first')
             .drop_duplicates(subset=['cik', 'year'], keep='last')
             .reset_index(drop=True)
    )

    # ── 2. Clean (vectorised pandas) ─────────────────────────────────────────
    for sec in SECTIONS:
        batch[f'{sec}_clean'] = clean_series(batch[sec])
    batch.drop(columns=SECTIONS, inplace=True)
    gc.collect()

    # ── 3. Stem (PyStemmer C bindings) ───────────────────────────────────────
    for sec in SECTIONS:
        batch[f'{sec}_stemmed'] = stem_series(batch[f'{sec}_clean'])

    # ── 4. Save preprocessed batch to local temp (atomic) ────────────────────
    pre_cols = (
        ['cik', 'year', 'filing_date']
        + [f'{s}_clean'   for s in SECTIONS]
        + [f'{s}_stemmed' for s in SECTIONS]
    )
    batch[pre_cols].to_parquet(pre_tmp, index=False, compression='snappy')

    # Drop stemmed columns — not needed for chunking
    batch.drop(columns=[f'{s}_stemmed' for s in SECTIONS], inplace=True)
    gc.collect()

    # ── 5. Tokenise + save chunk batch to local temp ──────────────────────────
    chunk_rows: list[dict] = []
    for sec in SECTIONS:
        texts = batch[f'{sec}_clean'].tolist()
        ciks  = batch['cik'].tolist()
        years = batch['year'].tolist()

        for doc_start in range(0, len(texts), DOC_BATCH):
            enc = tokenizer(
                texts[doc_start : doc_start + DOC_BATCH],
                max_length=CHUNK_SIZE,
                stride=STRIDE,
                return_overflowing_tokens=True,
                truncation=True,
                padding=False,
                add_special_tokens=False,
            )
            ctr: dict[int, int] = defaultdict(int)
            for ids, si in zip(enc['input_ids'], enc['overflow_to_sample_mapping']):
                gi = doc_start + si
                chunk_rows.append({
                    'cik':       ciks[gi],
                    'year':      years[gi],
                    'section':   sec,
                    'chunk_idx': ctr[gi],
                    'text':      tokenizer.decode(ids, skip_special_tokens=True),
                    'n_tokens':  len(ids),
                })
                ctr[gi] += 1
            del enc

    if chunk_rows:
        chk_df         = pd.DataFrame(chunk_rows)
        chk_df['year'] = chk_df['year'].astype('Int64')
        chk_df.to_parquet(chk_tmp, index=False, compression='snappy')
        del chk_df, chunk_rows

    del batch, texts, ciks, years
    gc.collect()

print(f'Batch loop done. Temp files in {TEMP_DIR}/')

## 7. Merge temp batches → Drive

Run this cell **once all batches are done**. It streams each local temp file into
a single `ParquetWriter` and writes the result to Drive. Local temp files are kept
so you can re-run the merge without reprocessing.

In [ ]:
def merge_batch_files(pattern: str, out_path: str, label: str) -> None:
    files = sorted(_glob.glob(pattern))
    # Skip empty sentinels (batches with no valid data)
    files = [f for f in files if os.path.getsize(f) > 0]
    print(f'Merging {len(files)} {label} batch files → {os.path.basename(out_path)}')

    writer: pq.ParquetWriter | None = None
    try:
        for f in tqdm(files, desc=f'Merging {label}'):
            table = pq.read_table(f)
            if writer is None:
                writer = pq.ParquetWriter(out_path, table.schema, compression='snappy')
            writer.write_table(table)
            del table
    finally:
        if writer:
            writer.close()

    size_mb = os.path.getsize(out_path) / 1e6
    meta    = pq.read_metadata(out_path)
    print(f'  → {meta.num_rows:,} rows, {size_mb:.1f} MB')


merge_batch_files(f'{TEMP_DIR}/pre_*.parquet',    PRE_PATH,    'preprocessed')
merge_batch_files(f'{TEMP_DIR}/chk_*.parquet',    CHUNKS_PATH, 'chunks')
print('Done — outputs written to Drive.')

## 7. Quality-control checks

Loads the saved files from disk (small column subsets only) to avoid pulling the full
text back into RAM.

In [ ]:
import matplotlib.pyplot as plt

# Load only word-count proxy columns — avoids pulling full text into RAM
wc_cols = ['cik', 'year'] + [f'{s}_clean' for s in SECTIONS]
pre_sample = pd.read_parquet(PRE_PATH, columns=wc_cols)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, sec in zip(axes, SECTIONS):
    col = f'{sec}_clean'
    word_counts = pre_sample[col].str.split().str.len().dropna()
    ax.hist(word_counts.clip(upper=15_000), bins=40, color='#1f77b4', edgecolor='white', alpha=0.85)
    ax.axvline(word_counts.median(), color='red', linestyle='--',
               label=f'Median {word_counts.median():.0f}')
    ax.set_title(f'{sec} — word count')
    ax.set_xlabel('Words (capped at 15k)')
    ax.set_ylabel('# filings')
    ax.legend(fontsize=8)
    ax.grid(axis='y', linestyle='--', alpha=0.4)

plt.suptitle('Clean section lengths across all filings', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['output_folder'], 'section_length_dist.png'), dpi=150, bbox_inches='tight')
plt.show()
del pre_sample

# Empty-section audit and chunk stats
# Load only lightweight columns from each file

pre_wc   = pd.read_parquet(PRE_PATH,    columns=['cik', 'year'] + [f'{s}_clean' for s in SECTIONS])
chunk_meta = pd.read_parquet(CHUNKS_PATH, columns=['cik', 'year', 'section', 'chunk_idx', 'n_tokens'])

print('Empty / near-empty sections (< 50 words after cleaning):')
for sec in SECTIONS:
    col   = f'{sec}_clean'
    short = (pre_wc[col].str.split().str.len() < 50).sum()
    pct   = short / len(pre_wc) * 100
    print(f'  {col}: {short:>4} filings ({pct:.1f}%)')

print('\nChunk counts per section (max chunk_idx per doc):')
chunk_stats = (
    chunk_meta.groupby(['cik', 'year', 'section'])['chunk_idx']
              .max().reset_index()
              .groupby('section')['chunk_idx']
              .agg(['mean', 'std', 'max'])
)
print(chunk_stats.round(1).to_string())

print('\nToken distribution per chunk:')
print(chunk_meta['n_tokens'].describe().round(1).to_string())

del pre_wc, chunk_meta

print('\ntext_preprocessed.parquet → use in nlp_features.ipynb (LM dict, TF-IDF)')
print('text_chunks.parquet       → use in finbert_features.ipynb (FinBERT inference)')

In [ ]:
from collections import defaultdict

# Process each section in batches of DOC_BATCH documents.
# HuggingFace tokenizes the whole batch in Rust, then returns overflow chunks
# with overflow_to_sample_mapping telling us which source doc each chunk came from.
DOC_BATCH: int = 256

chunk_rows: list[dict] = []

for sec in SECTIONS:
    clean_col = f'{sec}_clean'
    texts  = preprocessed[clean_col].tolist()
    ciks   = preprocessed['cik'].tolist()
    years  = preprocessed['year'].tolist()

    for batch_start in tqdm(range(0, len(texts), DOC_BATCH), desc=f'Chunking {sec}'):
        batch_texts = texts[batch_start : batch_start + DOC_BATCH]
        batch_ciks  = ciks [batch_start : batch_start + DOC_BATCH]
        batch_years = years[batch_start : batch_start + DOC_BATCH]

        enc = tokenizer(
            batch_texts,
            max_length=CHUNK_SIZE,
            stride=STRIDE,
            return_overflowing_tokens=True,
            truncation=True,
            padding=False,
            add_special_tokens=False,
        )

        # overflow_to_sample_mapping[i] = index within this batch of the source doc
        chunk_counter: dict[int, int] = defaultdict(int)
        for ids, sample_idx in zip(enc['input_ids'], enc['overflow_to_sample_mapping']):
            chunk_rows.append({
                'cik':       batch_ciks[sample_idx],
                'year':      batch_years[sample_idx],
                'section':   sec,
                'chunk_idx': chunk_counter[sample_idx],
                'text':      tokenizer.decode(ids, skip_special_tokens=True),
                'n_tokens':  len(ids),
            })
            chunk_counter[sample_idx] += 1

chunks_df = pd.DataFrame(chunk_rows)
chunks_df['year'] = chunks_df['year'].astype('Int64')

chunks_path = os.path.join(CONFIG['output_folder'], 'text_chunks.parquet')
chunks_df.to_parquet(chunks_path, index=False, compression='snappy')

size_mb = os.path.getsize(chunks_path) / 1e6
print(f'Saved text_chunks.parquet  {size_mb:.1f} MB')
print(f'Total chunks: {len(chunks_df):,}')
print(chunks_df.groupby('section')['chunk_idx'].max().rename('max_chunk_idx_per_doc').to_string())
chunks_df.head(4)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, sec in zip(axes, SECTIONS):
    col = f'{sec}_clean'
    word_counts = preprocessed[col].str.split().str.len().dropna()
    ax.hist(word_counts.clip(upper=15_000), bins=40, color='#1f77b4', edgecolor='white', alpha=0.85)
    ax.axvline(word_counts.median(), color='red', linestyle='--',
               label=f'Median {word_counts.median():.0f}')
    ax.set_title(f'{sec} — word count distribution')
    ax.set_xlabel('Words (capped at 15k)')
    ax.set_ylabel('# filings')
    ax.legend(fontsize=8)
    ax.grid(axis='y', linestyle='--', alpha=0.4)

plt.suptitle('Clean section lengths across all filings', fontsize=13)
plt.tight_layout()
plt.savefig(
    os.path.join(CONFIG['output_folder'], 'section_length_dist.png'),
    dpi=150, bbox_inches='tight'
)
plt.show()